# 如何构建langchain agents

In [3]:
import getpass
import os
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_deepseek import ChatDeepSeek

deepseek_api_key = "sk-f46c7e2053764299ace45b6e3d98de77"
base_url = "https://api.deepseek.com"

# model = ChatOpenAI(model="gpt-4o")
if not os.getenv("DEEPSEEK_API_KEY"):
    os.environ["DEEPSEEK_API_KEY"] = deepseek_api_key

llm = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # other params...
)


@tool
def magic_function(input: int) -> int:
    """Applies a magic function to an input."""
    return input + 2


tools = [magic_function]


query = "what is the value of magic_function(77)?"


In [4]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant"),
        ("human", "{input}"),
        # Placeholders fill up a **list** of messages
        ("placeholder", "{agent_scratchpad}"),
    ]
)


agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools)

agent_executor.invoke({"input": query})

{'input': 'what is the value of magic_function(77)?',
 'output': 'The value of `magic_function(77)` is `79`.'}

In [5]:
from langgraph.prebuilt import create_react_agent

langgraph_agent_executor = create_react_agent(llm, tools)


messages = langgraph_agent_executor.invoke({"messages": [("human", query)]})
{
    "input": query,
    "output": messages["messages"][-1].content,
}

{'input': 'what is the value of magic_function(77)?',
 'output': 'The value of `magic_function(77)` is `79`.'}

In [7]:
message_history = messages["messages"]
message_history

[HumanMessage(content='what is the value of magic_function(77)?', additional_kwargs={}, response_metadata={}, id='9d79b44d-0083-4ff1-adb5-18cce48fecf1'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_0_56a1218b-9eb1-4d00-b376-46ed98cc4f0c', 'function': {'arguments': '{"input":77}', 'name': 'magic_function'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 114, 'total_tokens': 133, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 114}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache', 'id': 'e3964d86-2f6e-4f86-8503-c0a26fd2d60f', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--89894975-dd30-447a-89cd-f28b6ad4d300-0', tool_calls=[{'name': 'magic_function', 'args': {'input': 77}, 'id': 'call_0_56a1218b

In [9]:
new_query = "Pardon?"

In [10]:
messages = langgraph_agent_executor.invoke(
    {"messages": message_history + [("human", new_query)]}
)


In [11]:
{
    "input": new_query,
    "output": messages["messages"][-1].content,
}

{'input': 'Pardon?',
 'output': "Apologies for any confusion! The value of `magic_function(77)` is `79`. Let me know if you'd like further clarification or assistance!"}

## summary
也就是， langgraph_agent_executor.invoke({"messages": list of messages})
可以作为agent执行用户查询

# Prompt template
- 原本的langchain agents需要prompt template来控制agent
- langgraph react agent executor可以用以下方式达成
    1. 输入中写入system messgae
    2. 用system message 初始化agent
    3. 用一个message transform in the graph state的function初始化agent
    4. 用runnable转换message，初始化agent

In [12]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Respond only in Germany."),
        ("human", "{input}"),
        # Placeholders fill up a **list** of messages
        ("placeholder", "{agent_scratchpad}"),
    ]
)


agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools)
agent_executor.invoke({"input": query})

{'input': 'what is the value of magic_function(77)?',
 'output': 'Der Wert der `magic_function(77)` ist **79**.'}

## langgraph prebuild `create_react_agent`
- langgraph预先构建好的 create_react_agent没有将prompt template设置成参数，而是将prompt设置成参数
- 这样在llm call之前，改变graph state
    - 这个prompt可以是systemMessage,也可以是string,会被转成SystemMessage， 
    - 也可以是callable， which should take in full graph state
    - or a runnable, which should take in full graph state

In [21]:
from langchain_core.messages import SystemMessage
from langgraph.prebuilt import create_react_agent
system_message = "You are a helpful assistant. Respond only in Germany."

langgraph_agent_executor = create_react_agent(llm, tools, prompt=system_message)
result = langgraph_agent_executor.invoke({"messages": [("user",query)]})

In [22]:
result

{'messages': [HumanMessage(content='what is the value of magic_function(77)?', additional_kwargs={}, response_metadata={}, id='70dadba1-0fd8-4847-8cd1-be65b755ca48'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_0_01241381-1895-46bd-b259-b66d3e737396', 'function': {'arguments': '{"input":77}', 'name': 'magic_function'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 119, 'total_tokens': 138, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 64}, 'prompt_cache_hit_tokens': 64, 'prompt_cache_miss_tokens': 55}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache', 'id': 'bf259b98-3cdb-425d-a05f-649ad12596c3', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--1d3e0ea9-b458-4d43-8271-68d72c179bd1-0', tool_calls=[{'name': 'magic_function', 'args': {'input': 77}, 'id': '

In [23]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.prebuilt import create_react_agent
from langgraph.prebuilt.chat_agent_executor import AgentState

# prompt = ChatPromptTemplate.from_messages(
#     [
#         ("system", "You are a helpful assistant. Respond only in Spanish."),
#         ("placeholder", "{messages}"),
#         ("user", "Also say 'Pandamonium!' after the answer."),
#     ]
# )

# alternatively, this can be passed as a function, e.g.
# 注意这里需要是 AgentState
def prompt(state: AgentState):
    return (
        [SystemMessage(content="You are a helpful assistant. Respond only in Spanish.")] +
        state["messages"] +
        [HumanMessage(content="Also say 'Pandamonium!' after the answer.")]
    )


langgraph_agent_executor = create_react_agent(llm, tools, prompt=prompt)


messages = langgraph_agent_executor.invoke({"messages": [("human", query)]})


In [24]:
print(
    {
        "input": query,
        "output": messages["messages"][-1].content,
    }
)

{'input': 'what is the value of magic_function(77)?', 'output': 'El valor de `magic_function(77)` es **79**. ¡Pandamonium!'}


# Memory
add chat Memory so it can engage in a multi-turn conversation.

In [25]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

In [ ]:
# in langchain
memory = InMemoryChatMessageHistory(session_id="test08020942")
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        # First put the history
        ("placeholder", "{chat_history}"),
        # Then the new input
        ("human", "{input}"),
        # Finally the scratchpad
        ("placeholder", "{agent_scratchpad}"),
    ]
)


@tool
def magic_function(input: int) -> int:
    """Applies a magic function to an input."""
    return input + 2


tools = [magic_function]


agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools)

agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,
    # This is needed because in most real world scenarios, a session id is needed
    # It isn't really used here because we are using a simple in memory ChatMessageHistory
    lambda session_id: memory,
    input_messages_key="input",
    history_messages_key="chat_history",
)

config = {"configurable": {"session_id": "test08020942"}}
print(
    agent_with_chat_history.invoke(
        {"input": "Hi, I'm polly! What's the output of magic_function of 3?"}, config
    )["output"]
)
print("---")
print(agent_with_chat_history.invoke({"input": "Remember my name?"}, config)["output"])
print("---")
print(
    agent_with_chat_history.invoke({"input": "what was that output again?"}, config)[
        "output"
    ]
)

The output of the `magic_function` for the input 3 is 5! Nice to meet you, Polly!
---
Of course, Polly! I remember your name. How can I assist you today? 😊
---
The output of the `magic_function` for the input 3 was **5**! Let me know if you'd like to explore anything else, Polly. 😊


In [27]:
# in langgraph

from langgraph.checkpoint.memory import MemorySaver  # an in-memory checkpointer
from langgraph.prebuilt import create_react_agent

system_message = "You are a helpful assistant."
# This could also be a SystemMessage object
# system_message = SystemMessage(content="You are a helpful assistant. Respond only in Spanish.")

memory = MemorySaver()
langgraph_agent_executor = create_react_agent(
    llm, tools, prompt=system_message, checkpointer=memory
)

config = {"configurable": {"thread_id": "test08020942"}}
print(
    langgraph_agent_executor.invoke(
        {
            "messages": [
                ("user", "Hi, I'm Nancy! What's the output of magic_function of -2?")
            ]
        },
        config,
    )["messages"][-1].content
)
print("---")
print(
    langgraph_agent_executor.invoke(
        {"messages": [("user", "Remember my name?")]}, config
    )["messages"][-1].content
)
print("---")
print(
    langgraph_agent_executor.invoke(
        {"messages": [("user", "what was that output again?")]}, config
    )["messages"][-1].content
)

The output of the `magic_function` for the input `-2` is `0`. Nice to meet you, Nancy! Let me know if you'd like to explore anything else.
---
Of course, Nancy! I remember your name. How can I assist you further? 😊
---
The output of the `magic_function` for the input `-2` was `0`. Let me know if you'd like to try another input or need anything else, Nancy! 😊


In [28]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# model = ChatOpenAI(model="gpt-4o")


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human", "{input}"),
        # Placeholders fill up a **list** of messages
        ("placeholder", "{agent_scratchpad}"),
    ]
)


@tool
def magic_function(input: int) -> int:
    """Applies a magic function to an input."""
    return input + 2


tools = [magic_function]

agent = create_tool_calling_agent(llm, tools, prompt=prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools)

for step in agent_executor.stream({"input": query}):
    print(step)

{'actions': [ToolAgentAction(tool='magic_function', tool_input={'input': 77}, log="\nInvoking: `magic_function` with `{'input': 77}`\n\n\n", message_log=[AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_0_8a46eb91-f796-4132-b344-dd0837ab10e0', 'function': {'arguments': '{"input":77}', 'name': 'magic_function'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache'}, id='run--4c2db5d8-57a8-4a18-92f7-870530529066', tool_calls=[{'name': 'magic_function', 'args': {'input': 77}, 'id': 'call_0_8a46eb91-f796-4132-b344-dd0837ab10e0', 'type': 'tool_call'}], usage_metadata={'input_tokens': 114, 'output_tokens': 19, 'total_tokens': 133, 'input_token_details': {'cache_read': 64}, 'output_token_details': {}}, tool_call_chunks=[{'name': 'magic_function', 'args': '{"input":77}', 'id': 'call_0_8a46eb91-f796-4132-b344-dd0837ab10e0', 'index': 0, 'type': 'tool_

In [30]:
from langgraph.prebuilt import create_react_agent
from langgraph.prebuilt.chat_agent_executor import AgentState

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("placeholder", "{messages}"),
    ]
)

langgraph_agent_executor = create_react_agent(llm, tools, prompt=prompt)

for step in langgraph_agent_executor.stream(
    {"messages": [("human", query)]}, stream_mode="updates"
):
    print(step)

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_0_681ca10c-0095-49a8-88b9-e7821927e64f', 'function': {'arguments': '{"input":77}', 'name': 'magic_function'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 114, 'total_tokens': 133, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 64}, 'prompt_cache_hit_tokens': 64, 'prompt_cache_miss_tokens': 50}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache', 'id': '6bf46178-b8ec-48cb-a1a6-791dc8c1e79e', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--302d5680-daf2-487c-be9e-dd7efcb16e32-0', tool_calls=[{'name': 'magic_function', 'args': {'input': 77}, 'id': 'call_0_681ca10c-0095-49a8-88b9-e7821927e64f', 'type': 'tool_call'}], usage_metadata={'input_tokens': 114, 'output_tokens': 19, 'total_tokens': 13

# return_intermediate_steps

In [31]:
agent_executor = AgentExecutor(agent=agent, tools=tools, return_intermediate_steps=True)
result = agent_executor.invoke({"input": query})
print(result["intermediate_steps"])

[(ToolAgentAction(tool='magic_function', tool_input={'input': 77}, log="\nInvoking: `magic_function` with `{'input': 77}`\n\n\n", message_log=[AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_0_28eaec14-8762-40ed-be5d-05036313d130', 'function': {'arguments': '{"input":77}', 'name': 'magic_function'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache'}, id='run--0504490e-dacd-4c51-a02f-9dfe86a50dd0', tool_calls=[{'name': 'magic_function', 'args': {'input': 77}, 'id': 'call_0_28eaec14-8762-40ed-be5d-05036313d130', 'type': 'tool_call'}], usage_metadata={'input_tokens': 114, 'output_tokens': 21, 'total_tokens': 135, 'input_token_details': {'cache_read': 64}, 'output_token_details': {}}, tool_call_chunks=[{'name': 'magic_function', 'args': '{"input":77}', 'id': 'call_0_28eaec14-8762-40ed-be5d-05036313d130', 'index': 0, 'type': 'tool_call_chunk'

In [32]:
from langgraph.prebuilt import create_react_agent

langgraph_agent_executor = create_react_agent(llm, tools=tools)

messages = langgraph_agent_executor.invoke({"messages": [("human", query)]})

messages

{'messages': [HumanMessage(content='what is the value of magic_function(77)?', additional_kwargs={}, response_metadata={}, id='b0e1324c-c3d1-4a14-999b-01015793fcb8'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_0_0e3fb581-d20f-4e5a-9d3d-c504370fe025', 'function': {'arguments': '{"input":77}', 'name': 'magic_function'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 114, 'total_tokens': 133, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 64}, 'prompt_cache_hit_tokens': 64, 'prompt_cache_miss_tokens': 50}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_8802369eaa_prod0623_fp8_kvcache', 'id': 'bc638b68-b5c1-463a-8da6-92873d28ff70', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--265ed204-3679-464d-ab7e-e19d981211aa-0', tool_calls=[{'name': 'magic_function', 'args': {'input': 77}, 'id': '